# 01. Data Cleaning & Feature Engineering

## Project: Statistical Analysis of Digital Habits & Mental Wellbeing
This notebook prepares the raw Social Media & Mental Health survey dataset (`N = 481` responses) for statistical inference. We standardize verbose survey headers, normalize inconsistent categorical text, inspect statistical outliers, and engineer numeric screen-time variables alongside a composite mental distress index.


In [1]:
import os
import pandas as pd
import numpy as np

# File paths
RAW_DATA_PATH = '../data/raw/wellbeing_data.csv'  # <-- UPDATE WITH YOUR FILENAME
PROCESSED_DATA_PATH = '../data/processed/cleaned_wellbeing_data.csv'

# Ensure processed directory exists
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)


### Step 1: Load & Inspect Raw Dataset
Load the raw CSV file and audit its dimensions, column data types, and initial records.


In [2]:
df = pd.read_csv(RAW_DATA_PATH)

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("--- Data Types & Missing Values ---")
df.info()

display(df.head())


Dataset Shape: 481 rows, 21 columns

--- Data Types & Missing Values ---
<class 'pandas.DataFrame'>
RangeIndex: 481 entries, 0 to 480
Data columns (total 21 columns):
 #   Column                                                                                                                Non-Null Count  Dtype  
---  ------                                                                                                                --------------  -----  
 0   Timestamp                                                                                                             481 non-null    str    
 1   1. What is your age?                                                                                                  481 non-null    float64
 2   2. Gender                                                                                                             481 non-null    str    
 3   3. Relationship Status                                                                       

,Timestamp,1. What is your age?,2. Gender,3. Relationship Status,4. Occupation Status,5. What type of organizations are you affiliated with?,6. Do you use social media?,7. What social media platforms do you commonly use?,8. What is the average time you spend on social media every day?,9. How often do you find yourself using Social media without a specific purpose?,...,11. Do you feel restless if you haven't used Social media in a while?,"12. On a scale of 1 to 5, how easily distracted are you?","13. On a scale of 1 to 5, how much are you bothered by worries?",14. Do you find it difficult to concentrate on things?,"15. On a scale of 1-5, how often do you compare yourself to other successful people through the use of social media?","16. Following the previous question, how do you feel about these comparisons, generally speaking?",17. How often do you look to seek validation from features of social media?,18. How often do you feel depressed or down?,"19. On a scale of 1 to 5, how frequently does your interest in daily activities fluctuate?","20. On a scale of 1 to 5, how often do you face issues regarding sleep?"
0,4/18/2022 19:18:47,21.0,Male,In a relationship,University Student,University,Yes,"Facebook, Twitter, Instagram, YouTube, Discord...",Between 2 and 3 hours,5,...,2,5,2,5,2,3,2,5,4,5
1,4/18/2022 19:19:28,21.0,Female,Single,University Student,University,Yes,"Facebook, Twitter, Instagram, YouTube, Discord...",More than 5 hours,4,...,2,4,5,4,5,1,1,5,4,5
2,4/18/2022 19:25:59,21.0,Female,Single,University Student,University,Yes,"Facebook, Instagram, YouTube, Pinterest",Between 3 and 4 hours,3,...,1,2,5,4,3,3,1,4,2,5
3,4/18/2022 19:29:43,21.0,Female,Single,University Student,University,Yes,"Facebook, Instagram",More than 5 hours,4,...,1,3,5,3,5,1,2,4,3,2
4,4/18/2022 19:33:31,21.0,Female,Single,University Student,University,Yes,"Facebook, Instagram, YouTube",Between 2 and 3 hours,3,...,4,4,5,5,3,3,3,4,4,1


In [3]:
print(df.columns.tolist())

['Timestamp', '1. What is your age?', '2. Gender', '3. Relationship Status', '4. Occupation Status', '5. What type of organizations are you affiliated with?', '6. Do you use social media?', '7. What social media platforms do you commonly use?', '8. What is the average time you spend on social media every day?', '9. How often do you find yourself using Social media without a specific purpose?', '10. How often do you get distracted by Social media when you are busy doing something?', "11. Do you feel restless if you haven't used Social media in a while?", '12. On a scale of 1 to 5, how easily distracted are you?', '13. On a scale of 1 to 5, how much are you bothered by worries?', '14. Do you find it difficult to concentrate on things?', '15. On a scale of 1-5, how often do you compare yourself to other successful people through the use of social media?', '16. Following the previous question, how do you feel about these comparisons, generally speaking?', '17. How often do you look to seek

### Step 2: Standardize Column Names & Clean Categorical Text
* Rename all 21 long-form survey questions into clean, `snake_case` variable names (e.g., `worries_score`, `depression_score`, `sleep_issues_score`).
* Consolidate inconsistent free-text entries in `gender` (mapping `'Nonbinary '`, `'NB'`, and `'Non binary'` into `'Non-binary'`).


In [4]:
# Map long survey questions to clean, analysis-ready column names
column_mapping = {
    'timestamp': 'timestamp',
    '1_what_is_your_age': 'age',
    '2_gender': 'gender',
    '3_relationship_status': 'relationship_status',
    '4_occupation_status': 'occupation',
    '5_what_type_of_organizations_are_you_affiliated_with': 'organization',
    '6_do_you_use_social_media': 'uses_social_media',
    '7_what_social_media_platforms_do_you_commonly_use': 'platforms',
    '8_what_is_the_average_time_you_spend_on_social_media_every_day': 'screen_time_category',
    '9_how_often_do_you_find_yourself_using_social_media_without_a_specific_purpose': 'purposeless_use',
    '10_how_often_do_you_get_distracted_by_social_media_when_you_are_busy_doing_something': 'distracted_busy',
    '11_do_you_feel_restless_if_you_havent_used_social_media_in_a_while': 'restlessness',
    '12_on_a_scale_of_1_to_5_how_easily_distracted_are_you': 'easily_distracted',
    '13_on_a_scale_of_1_to_5_how_much_are_you_bothered_by_worries': 'worries_score',
    '14_do_you_find_it_difficult_to_concentrate_on_things': 'concentration_diff',
    '15_on_a_scale_of_15_how_often_do_you_compare_yourself_to_other_successful_people_through_the_use_of_social_media': 'social_comparison',
    '16_following_the_previous_question_how_do_you_feel_about_these_comparisons_generally_speaking': 'comparison_feeling',
    '17_how_often_do_you_look_to_seek_validation_from_features_of_social_media': 'validation_seeking',
    '18_how_often_do_you_feel_depressed_or_down': 'depression_score',
    '19_on_a_scale_of_1_to_5_how_frequently_does_your_interest_in_daily_activities_fluctuate': 'interest_fluctuation',
    '20_on_a_scale_of_1_to_5_how_often_do_you_face_issues_regarding_sleep': 'sleep_issues_score'
}

# Standardize first, then rename
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', '_', regex=True)
)
df = df.rename(columns=column_mapping)

# Clean messy gender text (e.g., 'Nonbinary ', 'NB', 'There are others???')
df['gender'] = df['gender'].str.strip().replace({
    'Nonbinary': 'Non-binary',
    'Non binary': 'Non-binary',
    'NB': 'Non-binary',
    'unsure': 'Other',
    'There are others???': 'Other'
})

print("Cleaned Columns:", df.columns.tolist())


Cleaned Columns: ['timestamp', 'age', 'gender', 'relationship_status', 'occupation', 'organization', 'uses_social_media', 'platforms', 'screen_time_category', 'purposeless_use', 'distracted_busy', 'restlessness', 'easily_distracted', 'worries_score', 'concentration_diff', 'social_comparison', 'comparison_feeling', 'validation_seeking', 'depression_score', 'interest_fluctuation', 'sleep_issues_score']


### Step 3: Remove Duplicates & Handle Missing Values
Check for exact duplicate submissions and verify completeness across all numeric Likert scales and categorical fields.


In [5]:
# 1. Drop exact duplicate rows
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_rows - len(df)} duplicate rows.")

# 2. Check missing values per column
missing = df.isnull().sum()
print("\nMissing values before cleaning:")
print(missing[missing > 0])

# 3. Drop rows where critical target/predictor columns are missing (if any)
# df = df.dropna(subset=['screen_time', 'stress_level'])

# 4. Impute remaining missing values (Median for numeric, Mode for categorical)
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")


Removed 0 duplicate rows.

Missing values before cleaning:
organization    30
dtype: int64

Total missing values remaining: 0


### Step 4: Outlier Detection (IQR Method)
Evaluate continuous fields (such as `age`) using the Interquartile Range (IQR) rule to flag extreme values while preserving valid adult and senior survey responses.


In [6]:
def detect_outliers_iqr(dataframe, columns):
    outlier_summary = {}
    for col in columns:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = dataframe[(dataframe[col] < lower_bound) | (dataframe[col] > upper_bound)]
        outlier_summary[col] = {
            'outlier_count': len(outliers),
            'lower_bound': round(lower_bound, 2),
            'upper_bound': round(upper_bound, 2)
        }
    return pd.DataFrame(outlier_summary).T

print("--- IQR Outlier Summary ---")
display(detect_outliers_iqr(df, num_cols))

# Optional: Remove impossible values (e.g., screen time > 24 hours or negative age)
# df = df[(df['daily_screen_time'] >= 0) & (df['daily_screen_time'] <= 24)]


--- IQR Outlier Summary ---


,outlier_count,lower_bound,upper_bound
age,84.0,13.5,33.5
purposeless_use,20.0,1.5,5.5
distracted_busy,0.0,-1.0,7.0
restlessness,44.0,0.5,4.5
easily_distracted,33.0,1.5,5.5
worries_score,0.0,0.0,8.0
concentration_diff,0.0,-1.0,7.0
social_comparison,0.0,-1.0,7.0
comparison_feeling,33.0,0.5,4.5
validation_seeking,0.0,-2.0,6.0


### Step 5: Feature Engineering
Construct four core analytical variables for downstream hypothesis testing:
1. **`daily_screen_time`:** Map categorical usage brackets into numeric midpoint hours (`0.5`, `1.5`, `2.5`, `3.5`, `4.5`, `5.5`).
2. **`usage_tier`:** Group respondents into `Low (<2h)`, `Moderate (2-5h)`, and `High (>5h)` tiers for non-parametric group comparisons.
3. **`mental_distress_score`:** Sum five 1–5 Likert wellbeing items (`worries_score`, `concentration_diff`, `depression_score`, `interest_fluctuation`, `sleep_issues_score`) to create a composite scale ranging from **5 to 25**.
4. **`platform_count`:** Count the total number of distinct platforms each respondent actively uses.


In [9]:
# 1. Map text categories to numeric midpoint hours (for correlation & regression)
time_to_hours = {
    'Less than an Hour': 0.5,
    'Between 1 and 2 hours': 1.5,
    'Between 2 and 3 hours': 2.5,
    'Between 3 and 4 hours': 3.5,
    'Between 4 and 5 hours': 4.5,
    'More than 5 hours': 5.5
}
df['daily_screen_time'] = df['screen_time_category'].map(time_to_hours)

# 2. Group into 3 clean tiers (for ANOVA / Kruskal-Wallis)
tier_mapping = {
    'Less than an Hour': 'Low (<2h)',
    'Between 1 and 2 hours': 'Low (<2h)',
    'Between 2 and 3 hours': 'Moderate (2-5h)',
    'Between 3 and 4 hours': 'Moderate (2-5h)',
    'Between 4 and 5 hours': 'Moderate (2-5h)',
    'More than 5 hours': 'High (>5h)'
}
df['usage_tier'] = df['screen_time_category'].map(tier_mapping)

# 3. Create a composite Mental Wellbeing Distress Score (sum of key 1-5 wellbeing scales)
wellbeing_items = [
    'worries_score',
    'concentration_diff',
    'depression_score',
    'interest_fluctuation',
    'sleep_issues_score'
]
df['mental_distress_score'] = df[wellbeing_items].sum(axis=1)

# 4. Count number of platforms used per person
df['platform_count'] = df['platforms'].apply(lambda x: len(str(x).split(',')) if pd.notnull(x) else 0)

print("Usage Tier Counts:\n", df['usage_tier'].value_counts())
display(df[['daily_screen_time', 'usage_tier', 'mental_distress_score', 'sleep_issues_score', 'depression_score']].head())


Usage Tier Counts:
 usage_tier
Moderate (2-5h)    261
High (>5h)         116
Low (<2h)          104
Name: count, dtype: int64


,daily_screen_time,usage_tier,mental_distress_score,sleep_issues_score,depression_score
0,2.5,Moderate (2-5h),21,5,5
1,5.5,High (>5h),23,5,5
2,3.5,Moderate (2-5h),20,5,4
3,5.5,High (>5h),17,2,4
4,2.5,Moderate (2-5h),19,1,4


### Step 6: Export Processed Dataset
Save the cleaned, analysis-ready dataframe to `data/processed/cleaned_wellbeing_data.csv`.


In [10]:
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"✅ Cleaned dataset successfully saved to: {PROCESSED_DATA_PATH}")
print(f"Final Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")


✅ Cleaned dataset successfully saved to: ../data/processed/cleaned_wellbeing_data.csv
Final Dataset Shape: 481 rows, 25 columns
